# Metabis Case Study: Compass Th17 Reproduction

This notebook applies the staged Metabis workflow directly to the Th17 example used in the publication and in the Metabis server tutorial.

It assumes you already know the core objects and analysis steps from demo notebooks `01` to `04`. The focus here is not to reintroduce the API, but to reproduce the biological case study in one ordered run.

Case-study sequence:
- load the packaged Human1 model and bundled Th17 reaction and expression data
- define the Th17n and Th17p groups and initialize within-group analysis state
- run the Th17n versus Th17p comparison
- inspect global diagnostics
- interpret the result on the global network and selected subsystems


## Before you run it

This notebook assumes the repository dependencies are already installed.

When it is opened from a Metabis repository checkout, it adds the local `src/` directory automatically and loads the bundled Th17 example files from `metabis/data`.

The workflow below mirrors the publication/server case study directly: grouped data setup, within-group analysis, differential comparison, diagnostics, and pathway interpretation.


In [ ]:
import os
import sys
import tempfile
import warnings
from pathlib import Path

from bokeh.io import output_notebook


os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "metabis-matplotlib"),
)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", message="IProgress not found.*")


def find_metabis_repo_root_for_import(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]

    for candidate in [*direct_candidates, *nested_candidates]:
        if (candidate / "src" / "metabis").exists():
            return candidate

    raise FileNotFoundError("Could not locate the packaged Metabis repository root.")


repo_root_for_import = find_metabis_repo_root_for_import(Path.cwd().resolve())
src_path = repo_root_for_import / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from metabis import (
    DataManager,
    DataManagerMode,
    ExperimentAnalysis,
    ExpressionData,
    GroupComparison,
    MetabolicModel,
    ReactionData,
    plot_metabolic_graph,
    plot_reduced_dimension,
    plot_statistical_histograms,
    plot_subsystem_comparison,
)
from metabis.version import __version__ as metabis_version

output_notebook()


In [ ]:
def find_metabis_repo_root(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]

    for candidate in [*direct_candidates, *nested_candidates]:
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "demo").exists()
            and (candidate / "src" / "metabis").exists()
        ):
            return candidate

    raise FileNotFoundError("Could not locate the packaged Metabis repository root.")


def resolve_example_data_dir(repo_root: Path) -> Path:
    data_dir = repo_root / "data"

    if (
        (data_dir / "Th17_Compass_reactions.csv").exists()
        and (data_dir / "Th17_Compass_expression.csv").exists()
    ):
        return data_dir

    raise FileNotFoundError(
        "Could not find the bundled Th17 Compass example files in metabis/data."
    )


repo_root = find_metabis_repo_root(Path.cwd().resolve())
data_dir = resolve_example_data_dir(repo_root)
reaction_data_path = data_dir / "Th17_Compass_reactions.csv"
expression_data_path = data_dir / "Th17_Compass_expression.csv"

analysis_group_1 = "Th17n"
analysis_group_2 = "Th17p"
analysis_settings = {
    "pval_threshold": 0.10,
    "cohens_d_threshold": 0.0,
    "log2_fc_threshold": 0.0,
    "mean_difference_threshold": 0.0,
    "keep_reversed": True,
}

print(f"Metabis version: {metabis_version}")
print("Using bundled Th17 Compass example data")
print(f"Comparison direction: {analysis_group_1} -> {analysis_group_2}")
print(f"Adjusted p-value threshold: {analysis_settings['pval_threshold']}")


In [ ]:
model = MetabolicModel(use_cache=True)

alignment_file_path = model.get_alignment_file_path("RECON2")
model.apply_alignment_corrections(alignment_file_path)

print(
    f"Model loaded: {len(model.get_reactions())} reactions and {len(model.get_genes())} genes"
)


## Load the Th17 data

This case study uses the same loading configuration as the staged demos and moves directly into the Th17 application.


In [ ]:
reaction_loading_options = {
    "data_mode": "dual",
    "reaction_name_format": "rxnRecon3DID",
    "csv_separator": "\t",
    "samples_in": "columns",
    "check_old_identifiers": True,
}

expression_loading_options = {
    "gene_format": "geneSymbols",
    "normalization_options": ["CPM"],
    "data_format": "csv",
    "csv_separator": "\t",
    "samples_in": "columns",
}


In [ ]:
reaction_data, _ = ReactionData.create_with_validation(
    str(reaction_data_path),
    metabolic_model=model,
    **reaction_loading_options,
)

reaction_df = reaction_data.get_reaction_df()

print(
    f"Reaction matrix: {reaction_df.shape[0]} samples x {reaction_df.shape[1]} reaction features"
)


In [ ]:
expression_data, _ = ExpressionData.create_with_validation(
    str(expression_data_path),
    gene_list=model.get_genes(),
    metabolic_model=model,
    **expression_loading_options,
)

expression_df = expression_data.get_expression_df()

print(
    f"Expression matrix: {expression_df.shape[0]} samples x {expression_df.shape[1]} genes"
)


In [ ]:
dm = DataManager(model, reaction_data, expression_data, None)

dm.load_groups(criteria="sample_prefix")
group_ids = sorted(dm.get_group_ids())
sample_counts = {group_id: len(dm.get_group_samples(group_id)) for group_id in group_ids}

assert analysis_group_1 in group_ids and analysis_group_2 in group_ids, (
    f"Expected groups {analysis_group_1} and {analysis_group_2}, found {group_ids}"
)

print(f"Detected groups: {group_ids}")
print(f"Samples per group: {sample_counts}")


## Group structure before comparison

A quick reaction-space PCA is a useful bridge between grouped data setup and the differential comparison.


In [ ]:
analysis = ExperimentAnalysis(dm)
analysis.run_analysis(
    include_pca=True,
    pca_data_types=["reactions"],
    n_components=4,
)

dm.update_experiment_analysis(analysis)
dm.set_mode(DataManagerMode.EXPERIMENT_ANALYSIS)
dm.set_analysis_group(analysis_group_1)

print(f"Prepared experiment analysis for active group: {dm.get_active_group()}")


In [ ]:
plot_reduced_dimension(
    dm,
    method="PCA",
    data_type="reactions",
    figsize=(950, 700),
    n_variable_features=5000,
    return_plot=False,
)

## Group comparison: Th17n vs Th17p

This step applies the differential comparison directly, using a Wilcoxon test with Benjamini-Hochberg correction and the same significance settings used in the staged demos.


In [ ]:
dm.set_analysis_group(analysis_group_1)

group_comparison = GroupComparison(
    analysis_group_1,
    analysis_group_2,
    test="wilcoxon",
    correction="fdr_bh",
)

group_comparison.run_test(dm, for_genes=False)
group_comparison.run_test(dm, for_genes=True)


In [ ]:
significant_reactions, _ = group_comparison.define_significant_elements(
    pval_threshold=analysis_settings["pval_threshold"],
    cohens_d_threshold=analysis_settings["cohens_d_threshold"],
    log2_fc_threshold=analysis_settings["log2_fc_threshold"],
    mean_difference_threshold=analysis_settings["mean_difference_threshold"],
    keep_reversed=analysis_settings["keep_reversed"],
    for_genes=False,
)
significant_genes, _ = group_comparison.define_significant_elements(
    pval_threshold=analysis_settings["pval_threshold"],
    cohens_d_threshold=analysis_settings["cohens_d_threshold"],
    log2_fc_threshold=analysis_settings["log2_fc_threshold"],
    mean_difference_threshold=analysis_settings["mean_difference_threshold"],
    keep_reversed=analysis_settings["keep_reversed"],
    for_genes=True,
)


In [ ]:
dm.update_group_comparison(group_comparison)
dm.set_mode(DataManagerMode.GROUP_COMPARISON)
dm.set_analysis_group(analysis_group_1)

reaction_results = group_comparison.get_test_results(for_genes=False)
gene_results = group_comparison.get_test_results(for_genes=True)

print(f"Active analysis group: {dm.get_active_group()}")
print(f"Reaction-level results: {reaction_results.shape[0]} rows")
print(f"Gene-level results: {gene_results.shape[0]} rows")
print(f"Significant reactions: {len(significant_reactions)}")
print(f"Significant genes: {len(significant_genes)}")
print(
    "Thresholds applied: "
    f"adjusted p <= {analysis_settings['pval_threshold']}, "
    f"|Cohen's d| >= {analysis_settings['cohens_d_threshold']}, "
    f"|log2 FC| >= {analysis_settings['log2_fc_threshold']}, "
    f"|mean difference| >= {analysis_settings['mean_difference_threshold']}"
)


## Global diagnostics

Before moving to pathway interpretation, inspect the overall distribution of the comparison statistics and the subsystem ranking derived from the current comparison.


In [ ]:
plot_statistical_histograms(
    dm,
    parameters_to_plot=["pval", "log2_fc", "cohens_d"],
    return_plot=False,
    show_thresholds=True,
    for_genes=False,
)

In [ ]:
subsystem_stats = dm.subsystem_analysis.rank_subsystems_by_delta_score(
    for_genes=False
)

plot_subsystem_comparison(
    dm,
    subsystem_stats=subsystem_stats,
    metric="cohens_d",
    width=900,
    return_plot=False,
)

## Global metabolic network

With the comparison ordered as `Th17n -> Th17p`, red indicates higher values in Th17p and blue indicates higher values in Th17n.


The overview should recover the same high-level polarity reported in the case study: glycolysis shifts toward Th17p, while fatty-acid-oxidation-associated regions remain stronger in Th17n.


In [ ]:
plot_metabolic_graph(
    None,
    use_human1_map=True,
    layout_engine="fdp",
    data_manager=dm,
    color_metric="cohens_d",
    sizing_mode="scale_both",
    height=850,
    aspect_ratio=2,
    zoom=1,
    graph_type="metabolic_network",
)

## Pathway view: Glycolysis / Gluconeogenesis

This subsystem should show the expected Th17p-enriched glycolytic signal.


In [ ]:
plot_metabolic_graph(
    "Tricarboxylic acid cycle and glyoxylate/dicarboxylate metabolism",
    use_human1_map=True,
    layout_engine="dot",
    data_manager=dm,
    color_metric="cohens_d",
    sizing_mode="scale_both",
    height=750,
    highlight_significant_reactions=False,
    highlight_significant_genes=False,
    aspect_ratio=2,
    zoom=1,
    graph_type="subsystem",
)

## Pathway view: the TCA cycle

The key pattern here is not uniform activation, but a split or "broken" TCA configuration across individual reactions.


In [ ]:
plot_metabolic_graph(
    "Arginine and proline metabolism",
    use_human1_map=True,
    layout_engine="dot",
    data_manager=dm,
    color_metric="cohens_d",
    sizing_mode="scale_both",
    height=850,
    highlight_significant_reactions=False,
    highlight_significant_genes=False,
    aspect_ratio=2,
    zoom=1,
    graph_type="subsystem",
)

The TCA view should display red segments separated by more neutral reactions, reproducing the graph-level signature described in the case study.

## Integrated multi-omics around putrescine

The final subsystem couples reaction-level effects with gene context in arginine and proline metabolism, with particular attention to the route from arginine toward putrescine.


In [ ]:
gene_metrics = [
    "adjusted_pval",
    "log2_fc",
    "cohens_d",
    "mean_difference",
]
gene_summary_rows = []

for gene_symbol in ["SAT1", "ODC1"]:
    ensembl_id = model.translate_gene(
        gene_symbol,
        input_format="geneSymbols",
        output_format="ensmbl_id",
    )
    if ensembl_id not in gene_results.index:
        raise KeyError(
            f"{gene_symbol} ({ensembl_id}) was not found in the gene-level test results"
        )

    gene_row = gene_results.loc[ensembl_id, gene_metrics].copy()
    gene_row["ensmbl_id"] = ensembl_id
    gene_row["is_significant"] = ensembl_id in group_comparison.get_significant_elements(
        for_genes=True
    )
    gene_row.name = gene_symbol
    gene_summary_rows.append(gene_row)

gene_summary = pd.DataFrame(gene_summary_rows)[
    [
        "ensmbl_id",
        "adjusted_pval",
        "log2_fc",
        "cohens_d",
        "mean_difference",
        "is_significant",
    ]
]
gene_summary

## Case-study findings

This notebook applies the same ordered Metabis workflow used in the staged demos to the Th17 publication and server example in one direct run.

For this dataset, the reproduced biological story is consistent with the case study: glycolysis is stronger in Th17p, the fatty-acid-oxidation neighborhood is stronger in Th17n, the TCA cycle is only partially activated, and the arginine-to-putrescine branch adds gene-level context through `SAT1` and `ODC1`.

The same example is available in the web application through the `Compass_Th17` dataset.
